# Forge-LLM — Phase F preflight (Kaggle T4)

Public Kaggle training notebook (differentiation move #1 per `forge_llm.md`). Fork, attach a T4 accelerator, set `WANDB_API_KEY` and `HF_TOKEN` as Kaggle Secrets, then **Run All**. This notebook closes the open ADRs in `docs/DECISIONS.md` (004, 005, 006, 008, 009, 010) by populating the measurements in `docs/preflight/phase_f_numbers.md`.

Phase E (M1–M12) is complete on `main`. **This notebook is preflight, not the full training run** — Phase G reuses the trainer with the budget from `configs/train_kaggle_t4.yaml`.

Reference contracts:
- Checkpoint format: `CLAUDE.md` §8 (9-key dict).
- Resume safety: `tests/test_resume.py::test_resume_safety_loss_curve_indistinguishable` (bit-exact on single-GPU fp32; T4 fp16 validated empirically here).
- Free-tier discipline: `CLAUDE.md` §9 — secrets via Kaggle Secrets, loud warning if absent (§11: no silent fallback).
- Measurement template: `docs/preflight/phase_f_numbers.md`.

In [ ]:
# Install runtime deps + clone the repo into /kaggle/working.
# The Kaggle base image already has torch + datasets; we pin to pyproject.toml ranges
# so a fresh fork is reproducible.
import os, sys, subprocess

PINS = [
    "torch>=2.2,<2.10", "numpy>=1.26,<3", "datasets>=2.18,<4",
    "wandb>=0.17,<1", "huggingface_hub>=0.23,<1", "pyyaml>=6,<7",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *PINS])

# Two ways to get forge_llm itself:
#   (a) From PyPI (once M12 is published):     pip install -q forge-llm
#   (b) From a clone of the repo (current path while the package is unpublished).
# We default to (b). Replace REPO_URL with your fork if you've published changes.
REPO_URL = "https://github.com/Krishnarjunj/Forge_LLM"
REPO_DIR = "/kaggle/working/Forge_LLM"
if not os.path.exists(REPO_DIR):
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR])
sys.path.insert(0, f"{REPO_DIR}/src")

import forge_llm
print("forge_llm:", forge_llm.__version__)

In [ ]:
# Environment probe — fail loudly if no CUDA (Phase F preflight is meaningless on CPU).
import platform, subprocess, torch

print("torch:        ", torch.__version__)
print("cuda build:   ", torch.version.cuda)
print("cuda runtime: ", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError(
        "Phase F preflight requires a CUDA GPU. On Kaggle: "
        "Settings → Accelerator → GPU T4 x1."
    )
print("device:       ", torch.cuda.get_device_name(0))
print("python:       ", platform.python_version())
print("--- nvidia-smi ---")
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

In [ ]:
# Read WANDB_API_KEY + HF_TOKEN from Kaggle Secrets. Loud warning (CLAUDE.md §11) on missing.
import logging, os

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
log = logging.getLogger("preflight")

try:
    from kaggle_secrets import UserSecretsClient  # type: ignore[import-not-found]
    _client = UserSecretsClient()
except ImportError:
    log.warning("Not on Kaggle (kaggle_secrets unavailable). Falling back to os.environ.")
    _client = None

def _load_secret(name: str) -> str | None:
    if _client is None:
        return os.environ.get(name)
    try:
        v = _client.get_secret(name)
        os.environ[name] = v
        return v
    except Exception as e:  # noqa: BLE001 — kaggle_secrets exception classes vary; we log the type below
        log.warning("%s not set in Kaggle Secrets (%s). Continuing.", name, type(e).__name__)
        return None

WANDB_API_KEY = _load_secret("WANDB_API_KEY")
HF_TOKEN = _load_secret("HF_TOKEN")
print("wandb:  ", "configured" if WANDB_API_KEY else "DISABLED (CSV fallback)")
print("hf hub: ", "configured" if HF_TOKEN else "DISABLED (no checkpoint push)")

## Tokenizer

The repo commits `configs/tokenizer.json` when it exists; otherwise train a fresh one from a small FineWeb-Edu slice. Either way, the tokenizer is the single source of truth for the BPE; once written, every downstream cell reads it from disk.

In [ ]:
from pathlib import Path
from forge_llm.tokenizer import BPETokenizer

TOK_PATH = Path("/kaggle/working/Forge_LLM/configs/tokenizer.json")
if TOK_PATH.exists():
    tokenizer = BPETokenizer.load(TOK_PATH)
    print(f"loaded committed tokenizer: vocab_size={tokenizer.vocab_size}")
else:
    print("no committed tokenizer.json — training fresh from 5k FineWeb-Edu docs")
    sys.path.insert(0, "/kaggle/working/Forge_LLM/scripts")
    from train_bpe import _stream_fineweb, _train  # noqa: PLC0415
    _train(
        _stream_fineweb(5000, "HuggingFaceFW/fineweb-edu", "sample-10BT"),
        vocab_size=16384,
        out=TOK_PATH,
    )
    tokenizer = BPETokenizer.load(TOK_PATH)
    print(f"trained fresh: vocab_size={tokenizer.vocab_size}")

## Gate 1 — overfit one batch

Tile a single batch on `forge-30m`, train ~200 steps, expect loss to collapse toward 0. Catches a broken loss / data path / optim setup. Per `docs/preflight/phase_f_numbers.md` §2 Gate 1.

**Pass criterion:** final loss < 1.0 on a tiled corpus. If FAIL: stop, do not proceed. Open a debug ADR.

In [ ]:
import torch
from forge_llm.config import PRESETS, ForgeConfig
from forge_llm.train import Trainer, TrainConfig

g1_model_config = ForgeConfig(**PRESETS["forge-30m"])
g1_train_config = TrainConfig(
    lr=3e-4, micro_bs=4, grad_accum=1, seq_len=256,
    warmup_steps=0, total_steps=200, min_lr=3e-4,
    seed=0, dtype="fp16",
)

# Tile a single short corpus as the entire "dataset".
_g1_corpus = ["Forge-LLM overfit-one-batch sanity gate. " * 64] * 8
def _g1_factory():
    return iter(_g1_corpus)

g1_trainer = Trainer(
    g1_model_config, g1_train_config,
    data_factory=_g1_factory, tokenizer=tokenizer, device="cuda",
)
g1_losses = g1_trainer.train_steps(200)
print(f"initial loss: {g1_losses[0]:.3f}")
print(f"final loss:   {g1_losses[-1]:.3f}")
print("PASS" if g1_losses[-1] < 1.0 else "FAIL — model not overfitting a tiled batch")

## Gates 2–4 — 100-step smoke run on FineWeb-Edu

Real data path, real tokenizer, real model. Measure (per `docs/preflight/phase_f_numbers.md` §2 Gates 2–4):

- **tokens/sec** — mean over steps 50..100 (skip the first 50 as warmup).
- **peak GPU memory** — `torch.cuda.max_memory_allocated()`.
- **gpu_util** — sampled via `pynvml` during the run.

Numbers close ADR-004 (gpu_util threshold), ADR-005 (mem at n_layer=6), ADR-008 (micro_bs/grad_accum), ADR-009 (token budget).

In [ ]:
import time, torch
from forge_llm.config import PRESETS, ForgeConfig
from forge_llm.data import fineweb_doc_iterator
from forge_llm.train import Trainer, TrainConfig

# Wandb (optional)
wandb_run = None
if WANDB_API_KEY:
    import wandb  # noqa: PLC0415
    wandb_run = wandb.init(
        project="forge-llm",
        name="phase-f-preflight",
        config={"phase": "F", "n_layer": 6, "micro_bs": 8, "grad_accum": 16, "seq_len": 1024},
    )
else:
    print("WARNING: wandb disabled, metrics printed inline only.")

model_config = ForgeConfig(**PRESETS["forge-30m"])
train_config = TrainConfig(
    lr=3e-4, micro_bs=8, grad_accum=16, seq_len=1024,
    warmup_steps=100, total_steps=8000, min_lr=3e-5,
    seed=0, dtype="fp16",
)

def _factory():
    return fineweb_doc_iterator()

torch.cuda.reset_peak_memory_stats()
trainer = Trainer(
    model_config, train_config,
    data_factory=_factory, tokenizer=tokenizer, device="cuda",
)

# pynvml for gpu_util (Kaggle base image ships it; soft-fail otherwise).
try:
    import pynvml  # noqa: PLC0415
    pynvml.nvmlInit()
    _nvml_handle = pynvml.nvmlDeviceGetHandleByIndex(0)
except ImportError:
    pynvml = None
    _nvml_handle = None

losses, step_times, util_samples = [], [], []
for step in range(100):
    t0 = time.perf_counter()
    loss = trainer.train_steps(1)[0]
    torch.cuda.synchronize()
    step_times.append(time.perf_counter() - t0)
    losses.append(loss)
    if _nvml_handle is not None:
        util_samples.append(pynvml.nvmlDeviceGetUtilizationRates(_nvml_handle).gpu)
    if wandb_run:
        wandb.log({"train/loss": loss, "train/step_seconds": step_times[-1]}, step=step)
    if step % 10 == 0:
        print(f"step {step:3d}: loss={loss:.3f}, sec/step={step_times[-1]:.2f}")

# Headline numbers (steps 50..100, post-warmup).
steady_times = step_times[50:100]
tokens_per_step = train_config.micro_bs * train_config.grad_accum * train_config.seq_len
mean_sec = sum(steady_times) / len(steady_times)
tokens_per_sec = tokens_per_step / mean_sec
peak_mem_gb = torch.cuda.max_memory_allocated() / (1024**3)
mean_util = sum(util_samples[50:100]) / max(1, len(util_samples[50:100])) if util_samples else None

print()
print("=== Phase F headline numbers ===")
print(f"tokens/sec (mean steps 50..100):  {tokens_per_sec:9.1f}")
print(f"peak GPU memory (GB):              {peak_mem_gb:9.2f}")
print(f"gpu_util mean steps 50..100 (%):   {mean_util if mean_util is not None else '<no pynvml>'}")
print(f"mean sec/step (steps 50..100):     {mean_sec:9.3f}")
print("=================================")
if wandb_run:
    wandb.log({
        "preflight/tokens_per_sec": tokens_per_sec,
        "preflight/peak_gpu_mem_gb": peak_mem_gb,
        "preflight/gpu_util_mean_pct": mean_util or 0.0,
    })

## Gate 5 — hard-kill resume

Save a checkpoint, kill the kernel, restart, resume, and verify the loss curve matches a reference. The slow CPU test `tests/test_resume.py` already validates bit-exact resume at fp32; this gate validates the real Kaggle T4 fp16 environment (tolerance `atol≤1e-4` per `CLAUDE.md` §7).

**Manual step.** Run cell **A** (saves checkpoint, prints reference loss curve). Then `Kernel → Restart`. Then run the install / env / secrets / tokenizer cells above. Then run cell **B** (resume + compare). Do not run A and B back-to-back.

In [ ]:
# --- Gate 5 cell A --- save checkpoint, print reference losses, then RESTART KERNEL.
import torch

# First take a checkpoint of the current trainer (from Gates 2–4), then run 50 more steps
# on the same trainer to record a "reference" loss curve we'll compare against on resume.
ckpt_path = trainer.save_checkpoint("/kaggle/working/preflight_step100.pt")
print(f"saved {ckpt_path} at step={trainer._state.step}")

ref_losses = trainer.train_steps(50)
print("reference losses (steps 101..150 of the uninterrupted run):")
print([f"{x:.4f}" for x in ref_losses])
print()
print(">>> RESTART THE KERNEL NOW, then re-run the install/env/secrets/tokenizer cells,")
print(">>> then run Gate 5 cell B below.")

In [ ]:
# --- Gate 5 cell B --- resume from checkpoint, train 50 steps, compare to reference.
# Paste the `ref_losses` list printed in cell A into REF_LOSSES below before running.
REF_LOSSES = []  # e.g. [3.2174, 3.1988, 3.1721, ...]
if not REF_LOSSES:
    raise RuntimeError("Paste the reference losses from Gate 5 cell A into REF_LOSSES first.")

from forge_llm.data import fineweb_doc_iterator
from forge_llm.train import Trainer

def _resume_factory():
    return fineweb_doc_iterator()

resumed = Trainer.load_checkpoint(
    "/kaggle/working/preflight_step100.pt",
    data_factory=_resume_factory,
    tokenizer=tokenizer,
    device="cuda",
)
print(f"resumed at step {resumed._state.step}")
resumed_losses = resumed.train_steps(50)
print("resumed losses:")
print([f"{x:.4f}" for x in resumed_losses])

max_diff = max(abs(a - b) for a, b in zip(REF_LOSSES, resumed_losses))
print(f"max abs diff vs reference: {max_diff:.6f}")
print("PASS" if max_diff < 1e-4 else "FAIL — resume drifts more than 1e-4 from reference")

## ADR-006 — `torch.compile` generate() bench

Per `docs/DECISIONS.md` ADR-006: default to compile if speedup ≥1.5× at all three ctxs **and** cold-start (compile included) <60 s. Otherwise opt-in via `--compile`.

We compile the model (`torch.compile(model)`) rather than the `generate` generator function, since compiling a Python generator is non-trivial; the model forward pass is where the hot kernels live.

In [ ]:
import time, torch
from forge_llm.config import PRESETS, ForgeConfig
from forge_llm.generation import generate
from forge_llm.model import ForgeForCausalLM

cfg = ForgeConfig(**PRESETS["forge-30m"])
model_raw = ForgeForCausalLM(cfg).to("cuda").eval()
# Share weights so the compiled run isn't biased by a different init.
model_compiled = ForgeForCausalLM(cfg).to("cuda").eval()
model_compiled.load_state_dict(model_raw.state_dict())
model_compiled = torch.compile(model_compiled, mode="reduce-overhead")

PROMPT = "Once upon a time"

def _drive(model, max_new):
    tokens = 0
    for _ in generate(
        model, tokenizer, PROMPT,
        max_new=max_new, seed=0, use_cache=True, device="cuda",
    ):
        tokens += 1
    return tokens

# Cold-start measurement: first call with compile included.
torch.cuda.synchronize()
cold0 = time.perf_counter()
_drive(model_compiled, 32)
torch.cuda.synchronize()
cold_seconds = time.perf_counter() - cold0
print(f"cold-start with compile (32 tokens, includes graph capture): {cold_seconds:.1f}s")

# Steady-state bench across context lengths.
results = {}
for ctx in (128, 512, 2048):
    # Warm both
    _drive(model_raw, 8)
    _drive(model_compiled, 8)
    torch.cuda.synchronize()

    t0 = time.perf_counter()
    for _ in range(3):
        _drive(model_raw, ctx)
        torch.cuda.synchronize()
    tps_raw = ctx / ((time.perf_counter() - t0) / 3)

    t0 = time.perf_counter()
    for _ in range(3):
        _drive(model_compiled, ctx)
        torch.cuda.synchronize()
    tps_cmp = ctx / ((time.perf_counter() - t0) / 3)

    results[ctx] = (tps_raw, tps_cmp, tps_cmp / tps_raw)
    print(f"ctx={ctx:4d}: raw={tps_raw:7.1f} tok/s | compiled={tps_cmp:7.1f} tok/s | speedup={tps_cmp/tps_raw:.2f}x")

speedups = [results[c][2] for c in (128, 512, 2048)]
verdict = "default-on (--no-compile opt-out)" if (min(speedups) >= 1.5 and cold_seconds < 60) else "opt-in (--compile)"
print(f"\nADR-006 verdict: {verdict}")

## Gate 6 — wandb config dump

Visit the wandb run URL printed above and confirm the dashboard shows all of:

- [ ] `git_sha`
- [ ] resolved model config (full `ForgeConfig` JSON)
- [ ] `torch.__version__`, `torch.version.cuda`
- [ ] GPU name + driver
- [ ] full `pip freeze`
- [ ] `config_hash` (SHA256 of resolved config dict)
- [ ] OS / Python version

Trainer records these in the checkpoint; this gate is the dashboard-side confirmation.

In [ ]:
if wandb_run:
    print(f"wandb run URL: {wandb_run.url}")
    print("Open it and tick the boxes in the cell above.")
else:
    print("wandb disabled; verify Gate 6 manually from CSV in runs/ when WANDB_API_KEY is set.")

## Write-up — paste into `docs/preflight/phase_f_numbers.md`

Copy the block below into `docs/preflight/phase_f_numbers.md` §1. Then close each open ADR in `docs/DECISIONS.md` per the resolution criteria in §3 of the same file. Each ADR closure is a separate commit with a **max-2-word message** (per `CLAUDE.md` §0). **Do not push.**

In [ ]:
gpu_util_str = f"{mean_util:.1f}" if mean_util is not None else "<no pynvml>"
print(f"""\
tokens/sec (mean, steps 50..100):           {tokens_per_sec:.1f}
peak GPU memory (GB):                       {peak_mem_gb:.2f}
gpu_util (mean last 50 steps, %):           {gpu_util_str}
generate compile speedup ctx=128:           {results[128][2]:.2f}x
generate compile speedup ctx=512:           {results[512][2]:.2f}x
generate compile speedup ctx=2048:          {results[2048][2]:.2f}x
generate cold-start with compile (s):       {cold_seconds:.1f}
""")